In [ ]:
import os
import pandas as pd
from datasets import Dataset, DatasetDict, Audio
import soundfile as sf
from pathlib import Path
from huggingface_hub import HfApi, create_repo
import librosa
import uuid


audio_dir = Path("dataset_backup/audio")
test_audio_dir = Path("dataset_backup/test_audio")
transcription_file = Path("dataset_backup/transcriptions.csv")
test_transcription_file = Path("dataset_backup/test_transcriptions.csv")

def load_data(audio_dir, transcription_file):
    df = pd.read_csv(transcription_file)
    audio_files = [str(file) for file in audio_dir.glob("*.wav")]
    audio_data = []

    for audio_file in audio_files:
        unique_id = str(uuid.uuid4())
        audio_data.append({
            "id": unique_id,
            "audio": audio_file,  
            "transcription": df[df["audio_file"] == os.path.basename(audio_file)]["transcription"].values[0]
        })

    dataset = Dataset.from_list(audio_data)
    dataset = dataset.cast_column("audio", Audio())  
    return dataset


train_dataset = load_data(audio_dir, transcription_file)
test_dataset = load_data(test_audio_dir, test_transcription_file)

new_dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})


print(new_dataset)


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

hf_token = os.getenv("HF_TOKEN")

In [ ]:
from huggingface_hub import login, HfApi
import datasets


login(token=hf_token)

repo_name = "ufcg-labmet-fala-texto-main"

api = HfApi()
api.create_repo(repo_id=repo_name, private=False)

output_dir = "./huggingface_datasets"
new_dataset.save_to_disk(output_dir)

dataset = datasets.load_from_disk(output_dir)
dataset.push_to_hub(repo_name, private=False)